# Chapitre 1.2 : les données

Un modèle de langage ne vaut que ce qu’il a lu. Dans ce carnet, tu regardes de près comment « ton IA » prépare ses textes : d’où ils viennent, ce qu’on jette (les textes trop courts, les doublons, le code dont la licence ne permet pas de le réutiliser) et ce qu’on met de côté pour vérifier le modèle. Tu le fais avec le vrai code de forge, le moteur de « ton IA », pas avec une copie simplifiée.

Ce carnet accompagne [la page du chapitre 1.2](https://chipat-neko.github.io/forge-ia/partie-1/1-2/) : tu peux la lire avant, ou en même temps.

**Ce que tu vas faire**

1. regarder le mélange des sources ;
2. lire les trois sources et leurs licences ;
3. nettoyer un texte, pas à pas ;
4. repérer les doublons ;
5. trier le code selon sa licence ;
6. mettre de côté les textes de validation ;
7. relever un défi, puis vérifier que ce carnet donne exactement les mêmes résultats que la page du site.

**Comment l’utiliser**

Lance les cellules de code de haut en bas : clique sur le bouton rond avec un triangle, à gauche de la cellule, ou tape Maj + Entrée. Tu peux aussi choisir « Exécution », puis « Tout exécuter ». Tu peux modifier le code et le relancer autant de fois que tu veux : tu ne peux rien casser.

Sur Colab, la lecture des sources passe par Internet : le carnet lit en flux 2 Mio de chaque source sur Hugging Face, sans rien télécharger en entier.

## Étape 0 : récupérer le code

Le code du projet est dans un dépôt GitHub public. La cellule suivante le télécharge avec `git clone`, puis indique à Python où trouver le projet et forge. Le nettoyage, les doublons, les licences et la validation sont ceux de forge ; le projet ne fait que les appeler, dans l’ordre. Pour lire les sources en flux, Colab a déjà la bibliothèque `datasets` : il n’y a rien à installer.

Elle définit aussi `nombre`, qui écrit les nombres à la française, avec une espace fine entre les milliers et une virgule, et `voir`, qui rend visibles les caractères qu’on ne voit pas : un retour à la ligne ou une espace insécable.

In [ ]:
import os
import sys
from pathlib import Path

# Sur Colab, on télécharge le dépôt public. Quand ce carnet est lancé par les tests
# du projet, la variable FORGE_IA_LOCAL donne le chemin d’un dépôt déjà présent :
# on l’utilise, sans rien télécharger.
depot_local = os.environ.get("FORGE_IA_LOCAL")
if depot_local:
    depot = Path(depot_local)
else:
    depot = Path("forge-ia")
    if not depot.exists():
        !git clone --depth 1 https://github.com/chipat-neko/forge-ia

projet = depot / "projets" / "1.2-donnees"
if not projet.exists():
    raise FileNotFoundError("Le dépôt n’a pas pu être téléchargé. Vérifie ta connexion, puis relance cette cellule.")

# Python cherche les modules dans les dossiers de sys.path : on y ajoute le projet et forge.
sys.path.insert(0, str(projet.resolve()))
sys.path.insert(0, str((depot / "projets" / "forge").resolve()))


def nombre(x, decimales=0):
    # Un nombre à la française : « :, » sépare les milliers par des virgules, à l’anglaise ;
    # on les remplace par l’espace fine (\u202f), puis le point par une virgule.
    texte = f"{x + 0.0:,.{decimales}f}"
    return texte.replace(",", "\u202f").replace(".", ",")


def voir(texte):
    # Ce qu’on ne voit pas s’écrit en toutes lettres : retour à la ligne, espaces insécables, contrôles.
    visibles = {"\n": "⏎", "\r": "␍", "\t": "⇥", "\u00a0": "[espace insécable]", "\u3000": "[espace idéographique]", "\u2028": "[séparateur de ligne]"}
    return "".join(visibles.get(c, c if c.isprintable() else f"[U+{ord(c):04X}]") for c in texte)


print("Le code du projet et de forge est prêt.")

## Étape 1 : le mélange

« ton IA » n’apprend pas sur une seule source. Elle apprend sur un **mélange** : 75 % de français du web, 10 % de Wikipédia en français et 15 % de code Python. Ces parts sont écrites une seule fois, dans forge (`MELANGE_PAR_DEFAUT`), et le téléchargement, le tokenizer et l’encodage les lisent là. La fonction `repartir` partage un volume entre les sources, sans perdre un seul octet.

In [ ]:
from forge.donnees.melange import MELANGE_PAR_DEFAUT, repartir

total = 300 * 1024**2  # 300 Mio de texte, l’échantillon de la documentation
parts = repartir(total)
print(f"Pour {nombre(total / 1024**2)}\u202fMio de texte, le mélange demande\u202f:")
for nom, part in MELANGE_PAR_DEFAUT.items():
    print(f"  {nom}\u202f: {nombre(100 * part)}\u202f%, soit {nombre(parts[nom] / 1024**2, 1)}\u202fMio")
print(f"La somme des parts rendues vaut exactement le total\u202f: {sum(parts.values()) == total}")

## Étape 2 : les trois sources et leurs licences

Chaque source dit où elle se lit (un dépôt Hugging Face), dans quelle colonne se trouve le texte, sous quelle licence elle est publiée, et quels documents on garde. Une licence n’est pas un détail : c’est elle qui dit si on a le droit de s’en servir, et à quelles conditions.

In [ ]:
from forge.donnees.sources import LICENCES_PERMISSIVES, SOURCES

for source in SOURCES.values():
    print(f"{source.nom}\u202f: dépôt {source.depot}, colonne «\u202f{source.colonne}\u202f»")
    print(f"  licence\u202f: {source.licence}")
    print(f"  texte de la licence\u202f: {source.url_licence}")
print()
print("Pour le code, chaque fichier garde la licence de son dépôt d’origine. On ne garde que celles-ci\u202f:")
print("  " + ", ".join(sorted(LICENCES_PERMISSIVES)))

## Étape 3 : le nettoyage, pas à pas

Un texte de la source n’est jamais gardé tel quel. Le nettoyage de forge enchaîne quatre gestes, toujours dans le même ordre :

1. les fins de ligne Windows (`\r\n`) deviennent des `\n` ;
2. les caractères de contrôle partent (sauf le retour à la ligne et la tabulation) ;
3. le texte est mis en forme NFC : une lettre et son accent écrits en deux morceaux n’en font plus qu’un ;
4. les blancs de la fin disparaissent.

Un texte de moins de 200 caractères, une fois nettoyé, est écarté : il n’apprendrait presque rien. La cellule suivante nettoie un texte piégé, et montre ce qui change.

In [ ]:
from forge.donnees.nettoyage import LONGUEUR_MINIMALE, nettoyer

piege = (
    "Un café, écrit « cafe\u0301 » avec un accent à part.\r\n"
    "Un contrôle\u0007 invisible, une tabulation\tet un retour chariot seul \r"
    + "Le texte continue pour atteindre la longueur minimale. " * 4
    + "\u00a0 \u3000\n"
)
propre = nettoyer(piege)
print(f"Avant\u202f: {nombre(len(piege))} caractères. Après\u202f: {nombre(len(propre))} caractères.")
print("Avant, les 120 premiers caractères\u202f:", voir(piege[:120]))
print("Après, les 120 premiers caractères\u202f:", voir(propre[:120]))
print("La fin a perdu ses blancs\u202f:", repr(propre[-14:]))
print(f"Seuil\u202f: {LONGUEUR_MINIMALE} caractères. Un texte plus court devient {nettoyer('Trop court.')}.")

Pour la suite, il faut des documents. Sur Colab, le carnet lit en flux 2 Mio de chaque source sur Hugging Face, avec le lecteur de forge. Quand il est lancé par les tests du projet (`FORGE_IA_LOCAL`), il lit à la place les **exemples du projet** : `exemples/web.jsonl` et `exemples/code.jsonl`, une vingtaine de documents écrits à la main, pour que tout tourne sans réseau.

In [ ]:
from forge.donnees.telechargement import lecteur_hugging_face

from donnees_pas_a_pas.trier import lire_jsonl


def lire_en_flux(source, octets):
    # Les documents de la source, jusqu’à ce que leur colonne de texte fasse `octets` octets.
    lus, taille = [], 0
    flux = lecteur_hugging_face(source)
    try:
        for ligne in flux:
            lus.append(ligne)
            taille += len(str(ligne.get(source.colonne, "")).encode("utf-8"))
            if taille >= octets:
                break
    finally:
        flux.close()
    return lus


if depot_local:
    print("Lecture locale\u202f: les exemples du projet (exemples/web.jsonl et exemples/code.jsonl), sans réseau.")
    documents = {
        "web_fr": list(lire_jsonl(projet / "exemples" / "web.jsonl")),
        "code_python": list(lire_jsonl(projet / "exemples" / "code.jsonl")),
    }
else:
    print("Lecture en flux sur Hugging Face\u202f: 2\u202fMio de chaque source, rien n’est téléchargé en entier.")
    documents = {nom: lire_en_flux(SOURCES[nom], 2 * 1024**2) for nom in SOURCES}

for nom, lignes in documents.items():
    print(f"  {nom}\u202f: {nombre(len(lignes))} documents lus")

## Étape 4 : les doublons

Le web se répète : une même page est copiée, republiée, ou presque. Apprendre dix fois le même texte, c’est lui donner dix fois trop d’importance. Pour les repérer, forge calcule l’**empreinte** du texte nettoyé : un SHA-1, une suite de 40 chiffres hexadécimaux qui ne dépend que du contenu. Deux textes qui se nettoient pareil ont la même empreinte, même si l’un avait des fins de ligne Windows et des espaces en trop. Le premier est gardé ; les suivants sont des doublons.

La fonction `trier` passe chaque document par les quatre étapes (licence, longueur, doublon, validation) et rend un verdict.

In [ ]:
from donnees_pas_a_pas.trier import RAISONS, compter_raisons, trier
from forge.donnees.nettoyage import empreinte

web = documents["web_fr"]
verdicts = list(trier(web, SOURCES["web_fr"]))
for i, (ligne, verdict) in enumerate(zip(web, verdicts), start=1):
    if verdict.raison == "gardé":
        detail = "empreinte " + verdict.empreinte[:8]
    elif verdict.raison == "doublon":
        detail = "même empreinte que " + verdict.empreinte[:8]
    else:
        detail = f"{len(ligne['text'])} caractères avant le nettoyage"
    if i <= 4 or verdict.raison == "doublon":
        print(f"document {i:>2}\u202f: {verdict.raison}, {detail}")

# Un texte, une copie avec des fins de ligne Windows, et une copie avec des blancs en trop.
texte = web[1]["text"]
print()
print("Même empreinte avec des fins de ligne Windows\u202f:", empreinte(texte.replace("\n", "\r\n")) == empreinte(texte))
print("Même empreinte avec des blancs en trop à la fin\u202f:", empreinte(texte + "  \n") == empreinte(texte))
print("Même empreinte avec une lettre changée\u202f:", empreinte(texte.replace("e", "é", 1)) == empreinte(texte))

## Étape 5 : la licence du code

Le code des dépôts publics n’est pas libre de droits : chaque fichier garde la licence de son dépôt. « ton IA » ne garde que les fichiers Python sous une licence permissive (MIT, Apache-2.0, BSD…), qui autorise à réutiliser le code en citant ses auteurs. Les autres sont écartés, pour l’une de deux raisons que le verdict distingue : le **langage** (un fichier JavaScript n’est pas du Python) ou la **licence** (un fichier Python sous GPL-3.0 n’est pas gardé).

In [ ]:
code_verdicts = list(trier(documents["code_python"], SOURCES["code_python"]))
# Deux exemples par raison au plus, pas une ligne par fichier\u202f: sur Colab, il y en a des centaines.
montres = {raison: 0 for raison in RAISONS}
for ligne, verdict in zip(documents["code_python"], code_verdicts):
    if montres[verdict.raison] < 2:  # deux exemples par raison au plus
        montres[verdict.raison] += 1
        print(f"{ligne.get('language')}, licence {ligne.get('license')}\u202f: {verdict.raison}")
print()
print("Raisons dans le code\u202f:", {raison: n for raison, n in compter_raisons(code_verdicts).items() if n})

## Étape 6 : la validation

On ne vérifie pas un modèle sur les textes qu’il a appris. Forge met de côté 0,5 % des documents, soit un sur 200, pour la **validation**. Pour que le choix ne dépende ni de l’ordre ni du moment du calcul, il se lit dans l’empreinte : les 8 premiers chiffres hexadécimaux, lus comme un nombre, sont comparés à 0,5 % du plus grand nombre possible. Le même document tombe donc toujours du même côté.

In [ ]:
from forge.donnees.melange import est_validation
from forge.donnees.nettoyage import empreinte

empreintes = [empreinte(f"Le document numéro {i} parle de forge et de données. " * 4) for i in range(20000)]
cote = [est_validation(e) for e in empreintes]
print(f"Sur {nombre(len(cote))} documents, {nombre(sum(cote))} vont en validation, soit {nombre(100 * sum(cote) / len(cote), 2)}\u202f%.")
# On recalcule tout\u202f: le même document retombe du même côté.
print("Le même document tombe du même côté à chaque calcul\u202f:", cote == [est_validation(e) for e in empreintes])

## Exercice : compter les raisons d’écart

Écris la fonction `compter_les_raisons(verdicts)`. Elle reçoit une liste de verdicts et rend un dictionnaire qui dit, pour chacune des cinq raisons (`gardé`, `trop court`, `doublon`, `licence` et `langage`), combien de verdicts l’ont. Les raisons qui n’apparaissent pas comptent zéro.

La cellule l’essaie sur les verdicts du web et du code, et la compare à `compter_raisons` du projet. Si les deux comptes sont les mêmes, le défi est réussi.

Indice : commence par un dictionnaire où chaque raison vaut 0 (`RAISONS` donne les cinq noms), puis parcours les verdicts et ajoute 1 à la raison de chacun.

In [ ]:
from donnees_pas_a_pas.trier import RAISONS, compter_raisons


def compter_les_raisons(verdicts):
    # À toi : un dictionnaire raison → nombre de verdicts, avec les cinq raisons.
    return {}


# Ne change rien en dessous : cette partie essaie ta fonction sur deux listes de verdicts.
reussis = 0
for nom, liste in [("web", verdicts), ("code", code_verdicts)]:
    voulu = compter_raisons(liste)
    obtenu = compter_les_raisons(liste)
    if obtenu == voulu:
        reussis += 1
    else:
        print(f"Pour le {nom}, ta fonction donne {obtenu} au lieu de {voulu}.")
if reussis == 2:
    print("Défi réussi\u202f: les deux comptes sont les bons.")
else:
    print(f"Pas encore ({reussis} liste sur 2).")

<details>
<summary>Voir une correction (essaie d’abord !)</summary>

On part d’un dictionnaire qui met chaque raison à zéro, pour que celles qui n’apparaissent pas soient comptées quand même. Puis chaque verdict ajoute 1 à sa raison.

Remplace la fonction de la cellule de l’exercice par :

```python
def compter_les_raisons(verdicts):
    comptes = {raison: 0 for raison in RAISONS}
    for verdict in verdicts:
        comptes[verdict.raison] += 1
    return comptes
```

</details>

## Dernière étape : les mêmes résultats que la page ?

La page du site n’utilise pas Python : elle refait le nettoyage en TypeScript, directement dans ton navigateur. Pour être sûr que les deux donnent exactement les mêmes résultats, le dépôt garde un fichier de référence, `tests/attendus.json`, calculé par le Python : les blancs que Python retire en fin de texte, le texte nettoyé, la longueur, l’empreinte et la place (entraînement ou validation) de textes piégés, le filtre des licences et les verdicts du tri sur les exemples. Les tests de la page relisent ce fichier. La cellule suivante recalcule tout avec les fonctions du projet, et compare chaque caractère.

In [ ]:
import json

from donnees_pas_a_pas.attendus import calculer_attendus, vers_json

attendus = json.loads((projet / "tests" / "attendus.json").read_text(encoding="utf-8"))
calcule = calculer_attendus()

print(
    f"{len(attendus['blancs_python'])} blancs de Python, {len(attendus['textes'])} textes de référence, "
    f"{len(attendus['licences'])} lignes de licence et {sum(len(t['verdicts']) for t in attendus['tris'])} verdicts comparés."
)
# Chaque section est écrite en JSON, comme dans le fichier, puis comparée caractère par caractère.
differentes = [nom for nom in attendus if vers_json(calcule[nom]) != vers_json(attendus[nom])]
for nom in differentes:
    print(f"La section «\u202f{nom}\u202f» n’est pas celle du fichier de référence.")

print(f"Même résultat que la page du site\u202f: {'oui' if not differentes else 'non'}")  # \u202f est l’espace fine insécable

## Pour aller plus loin

- Le code complet du projet, avec ses tests : [projets/1.2-donnees sur GitHub](https://github.com/chipat-neko/forge-ia/tree/main/projets/1.2-donnees).
- La page du chapitre, pour voir le tri des documents et lire le code expliqué : [chapitre 1.2](https://chipat-neko.github.io/forge-ia/partie-1/1-2/).
- Depuis le dossier du projet, `uv run python -m donnees_pas_a_pas trier exemples/web.jsonl` affiche un verdict par ligne, et `uv run python -m donnees_pas_a_pas apercu --octets 5M` lit 5 Mio de chaque source sur Hugging Face (il faut Internet).

Le code de ce carnet est sous licence MIT, ses textes sous licence CC BY 4.0.